In [0]:
# spark.conf.set("spark.checkpoint.dir", "/Volumes/databricks_practice/default/bronze/checkpoints")

In [0]:
# Country_code, Currency, Gender Segment tem nulos

In [0]:
import sys
sys.path.append("/Workspace/Shared/databricks-practice/utils/")

import pyspark.sql.functions as F

from functions import *

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS silver

In [0]:
bronze_table = "databricks_practice.bronze.bronze_table"
silver_table = "databricks_practice.silver.silver_table"
partition_col = "dt_ingestion_partition"


In [0]:
%sql
CREATE TABLE IF NOT EXISTS databricks_practice.silver.silver_table (
  price_local DECIMAL(10, 2),
  sale_price_local DECIMAL(10, 2),
  discount_pct DECIMAL(10, 2),
  size_count NUMERIC,
  available_market BOOLEAN,
  available_size_count NUMERIC,
  snapshot_date DATE,
  product_id STRING,
  product_name STRING,
  model_number STRING,
  style_color STRING,
  brand_name STRING,
  category STRING,
  subcategory STRING,
  color_name STRING,
  sport_tags STRING,
  product_url STRING,
  canonical_url STRING,
  image_url STRING,
  sku STRING,
  catalog_sku_id STRING,
  stock_keeping_unit_id STRING,
  country_code STRING,
  currency STRING,
  dt_ingestion_partition DATE,
  update_ts TIMESTAMP
) USING DELTA;




In [0]:
df_new = load_bronze_new_partitions(spark, bronze_table, silver_table, partition_col)

In [0]:
df_silver = clean_mercado(df_new)

In [0]:
display(df_silver_tratado)

In [0]:
df_silver_tratado = df_silver \
   .withColumn("price_local", F.col("price_local").cast("decimal(10,2)")) \
   .withColumn("sale_price_local", F.col("sale_price_local").cast("decimal(10,2)")) \
   .withColumn("discount_pct", F.col("discount_pct").cast("decimal(10,2)")) \
   .withColumn("size_count", F.col("size_count").cast("double")) \
   .withColumn("available_market", F.col("available_market").cast("boolean")) \
   .withColumn("available_size_count", F.col("available_size_count").cast("double")) \
   .withColumn("snapshot_date", F.col("snapshot_date").cast("date")) \
   .withColumn("product_id", F.col("product_id").cast("string")) \
   .withColumn("product_name", F.col("product_name").cast("string")) \
   .withColumn("model_number", F.col("model_number").cast("string")) \
   .withColumn("style_color", F.col("style_color").cast("string")) \
   .withColumn("brand_name", F.col("brand_name").cast("string")) \
   .withColumn("category", F.col("category").cast("string")) \
   .withColumn("subcategory", F.col("subcategory").cast("string")) \
   .withColumn("color_name", F.col("color_name").cast("string")) \
   .withColumn("sport_tags", F.col("sport_tags").cast("string")) \
   .withColumn("product_url", F.col("product_url").cast("string")) \
   .withColumn("canonical_url", F.col("canonical_url").cast("string")) \
   .withColumn("image_url", F.col("image_url").cast("string")) \
   .withColumn("sku", F.col("sku").cast("string")) \
   .withColumn("catalog_sku_id", F.col("catalog_sku_id").cast("string")) \
   .withColumn("stock_keeping_unit_id", F.col("stock_keeping_unit_id").cast("string")) \
   .withColumn("country_code", F.col("country_code").cast("string")) \
   .withColumn("currency", F.col("currency").cast("string")) \
   .withColumn("dt_ingestion_partition", F.col("dt_ingestion_partition").cast("date")) \
   .withColumn("update_ts", F.current_timestamp())

In [0]:
%sql
SELECT *
FROM (
  SELECT *,
    ROW_NUMBER() OVER(
      PARTITION BY product_id, sku, country_code, size_count, snapshot_date
      ORDER BY dt_ingestion DESC
    ) AS rn
  FROM bronze_table
)
WHERE rn = 1

In [0]:
# checkpointed_df = df_global.checkpoint()